# Import libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import scipy.stats as stat
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
import statsmodels.api as sm

# Import Data

In [ ]:
loan_data_preprocessed_backup = pd.read_csv('loan_data_preprocessed.csv')
loan_data_preprocessed = loan_data_preprocessed_backup.copy()

In [ ]:
print(loan_data_preprocessed.shape)
loan_data_preprocessed.head()

(466285, 323)


,Unnamed: 0.1,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,...,dti:22.4-35,dti:>35,mths_since_last_record:Missing,mths_since_last_record:0-2,mths_since_last_record:3-20,mths_since_last_record:21-31,mths_since_last_record:32-80,mths_since_last_record:81-86,mths_since_last_record:>86,good_bad
0,25916,25916,577956,743149,4800,4800,4800.0,36 months,7.14,148.52,...,0,0,1,0,0,0,0,0,0,1
1,83271,83271,7645235,9327358,18000,18000,18000.0,60 months,15.10,429.17,...,0,0,1,0,0,0,0,0,0,1
2,226146,226146,1144674,1385407,18000,18000,17975.0,36 months,12.12,598.89,...,0,0,1,0,0,0,0,0,0,1
3,424372,424372,12646890,14649041,21600,21600,21600.0,36 months,8.90,685.87,...,0,0,1,0,0,0,0,0,0,1
4,74133,74133,8117830,9859815,3300,3300,3300.0,36 months,15.61,115.39,...,0,0,1,0,0,0,0,0,0,1


In [ ]:
pd.options.display.max_rows=None
loan_data_preprocessed.isnull().sum()

Unnamed: 0.1                                                               0
Unnamed: 0                                                                 0
id                                                                         0
member_id                                                                  0
loan_amnt                                                                  0
funded_amnt                                                                0
funded_amnt_inv                                                            0
term                                                                       0
int_rate                                                                   0
installment                                                                0
grade                                                                      0
sub_grade                                                                  0
emp_title                                                              27588

In [ ]:
loan_data_preprocessed['mths_since_last_delinq'].fillna(0, inplace = True)
loan_data_preprocessed['mths_since_last_record'].fillna(0, inplace = True)

# LGD

In [ ]:
class LinearRegressionWithPValues(BaseEstimator, RegressorMixin):
    def __init__(self):
        self.model = None
        self.results = None
        self.coef_ = None
        self.intercept_ = None
        self.p_values_ = None
        self.r_squared_ = None
    def fit(self, X, y):
        # Add a constant term for the intercept
        X_with_intercept = sm.add_constant(X)

        # Fit the model using statsmodels
        self.model = sm.OLS(y, X_with_intercept)
        self.results = self.model.fit()

        # Store coefficients and p-values
        self.coef_ = self.results.params[1:]  # Exclude the intercept
        self.intercept_ = self.results.params[0]  # Intercept
        self.p_values_ = self.results.pvalues[1:]  # Exclude the intercept p-value
        self.r_squared_ = self.results.rsquared
        return self

    def predict(self, X):
        # Ensure to add the intercept column
        X_with_intercept = sm.add_constant(X, has_constant='add')
        return np.dot(X_with_intercept, self.results.params)

    def get_params(self, deep=True):
        return {}

    def set_params(self, **params):
        return self

    def get_p_values(self):
        return self.p_values_

    def get_r_squared(self):
        return self.r_squared_

In [ ]:
class LogisticRegression_with_p_values:

    def __init__(self,*args,**kwargs):#,**kwargs):
        self.model = linear_model.LogisticRegression(*args,**kwargs)#,**args)

    def fit(self,X,y):
        self.model.fit(X,y)

        #### Get p-values for the fitted model ####
        denom = (2.0 * (1.0 + np.cosh(self.model.decision_function(X))))
        denom = np.tile(denom,(X.shape[1],1)).T
        F_ij = np.dot((X / denom).T,X) ## Fisher Information Matrix
        Cramer_Rao = np.linalg.inv(F_ij) ## Inverse Information Matrix
        sigma_estimates = np.sqrt(np.diagonal(Cramer_Rao))
        z_scores = self.model.coef_[0] / sigma_estimates # z-score for eaach model coefficient
        p_values = [stat.norm.sf(abs(x)) * 2 for x in z_scores] ### two tailed test for p-values
        self.coef_ = self.model.coef_
        self.intercept_ = self.model.intercept_
        self.p_values = p_values


In [ ]:
reg_lgd_st_1 = pickle.load(open('lgd_model_stage_1.sav', 'rb'))
reg_lgd_st_2 = pickle.load(open('lgd_model_stage_2.sav', 'rb'))

In [ ]:
accepted_feature_lgd_1 = ['term_int', 'months_since_issue_d',
                    'months_since_earliest_cr_line',
                    'funded_amnt', 'installment',
                     'dti', 'mths_since_last_delinq','mths_since_last_record',
                     'total_acc', 'total_rev_hi_lim']

In [ ]:
accepted_feature_lgd_2 = ['grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'home_ownership:MORTGAGE',
'home_ownership:OWN',
'purpose:educational',
'purpose:moving',
'initial_list_status:w',
'months_since_issue_d',
'months_since_earliest_cr_line',
'int_rate',
'inq_last_6mths',
'open_acc',
'total_acc',
'total_rev_hi_lim']

In [ ]:
loan_data_preprocessed_lgd_1= loan_data_preprocessed.loc[:, accepted_feature_lgd_1]

In [ ]:
loan_data_preprocessed_lgd_2  = loan_data_preprocessed.loc[:, accepted_feature_lgd_2]

In [ ]:
loan_data_preprocessed['recovery_rate_st_1'] = reg_lgd_st_1.model.predict(loan_data_preprocessed_lgd_1)


In [ ]:
loan_data_preprocessed['recovery_rate_st_2'] = reg_lgd_st_2.predict(loan_data_preprocessed_lgd_2)


In [ ]:
loan_data_preprocessed['recovery_rate'] = loan_data_preprocessed['recovery_rate_st_1'] * loan_data_preprocessed['recovery_rate_st_2']


In [ ]:
loan_data_preprocessed['recovery_rate'] = np.where(loan_data_preprocessed['recovery_rate'] < 0, 0, loan_data_preprocessed['recovery_rate'])
loan_data_preprocessed['recovery_rate'] = np.where(loan_data_preprocessed['recovery_rate'] > 1, 1, loan_data_preprocessed['recovery_rate'])

In [ ]:
loan_data_preprocessed['LGD'] = 1 - loan_data_preprocessed['recovery_rate']

# EAD

In [ ]:
reg_ead = pickle.load(open('ead.sav', 'rb'))

In [ ]:
ead_accepted_features = ['grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'grade:F',
'home_ownership:MORTGAGE',
'verification_status:Source Verified',
'purpose:debt_consolidation',
'purpose:educational',
'purpose:home_improvement',
'purpose:major_purchase',
'purpose:medical',
'purpose:moving',
'purpose:other',
'purpose:renewable_energy',
'purpose:small_business',
'purpose:wedding',
'initial_list_status:w',
'term_int',
'emp_length_int',
'months_since_issue_d',
'months_since_earliest_cr_line',
'funded_amnt',
'int_rate',
'installment',
'dti',
'inq_last_6mths',
'mths_since_last_delinq',
'open_acc',
'total_acc']

In [ ]:
loan_data_preprocessed_ead =  loan_data_preprocessed.loc[:, ead_accepted_features]

In [ ]:
loan_data_preprocessed['CCF'] = reg_ead.predict(loan_data_preprocessed_ead)
loan_data_preprocessed['CCF'] = np.where(loan_data_preprocessed['CCF'] < 0, 0, loan_data_preprocessed['CCF'])
loan_data_preprocessed['CCF'] = np.where(loan_data_preprocessed['CCF'] > 1, 1, loan_data_preprocessed['CCF'])

In [ ]:
loan_data_preprocessed['EAD'] = loan_data_preprocessed['CCF'] * loan_data_preprocessed['funded_amnt']


# PD

In [ ]:
reg_pd = pickle.load(open('pd_model.sav', 'rb'))

In [ ]:
pd_input_with_cat =  ['grade:A',
'grade:B',
'grade:C',
'grade:D',
'grade:E',
'grade:F',
'grade:G',
'home_ownership:MORTGAGE',
'home_ownership:OWN',
'home_ownership:RENT_OTHER_NONE_ANY',
'verification_status:Not Verified',
'verification_status:Source Verified',
'verification_status:Verified',
'purpose:credit_card',
'purpose:debt_consolidation',
'addr_state:CA',
'addr_state:NY',
'addr_state:TX',
'initial_list_status:f',
'initial_list_status:w',
'addr_state:ND_NE_IA_NV_AL_FL',
'addr_state:MO_HI_NC_LA',
'addr_state:NH_AK_MS_WV_WY_DC_ME_ID',
'addr_state:CO_CT_SC_VT',
'addr_state:IL_KS',
'addr_state:OR_MT_WI',
'addr_state:GA_MN_IN_WA',
'addr_state:DE_MA_UT_KY',
'addr_state:PA_SD_OH',
'addr_state:NM_AZ_RI_MI_AR_TN',
'purpose:sm_b__edu__mov__other__house__med__re_e__wedding__vacation',
'purpose:vacation_major_purch__car__home_impr',
'term:36',
'term:60',
'emp_length:0',
'emp_length:1-4',
'emp_length:5-6',
'emp_length:7-9',
'emp_length:10',
'mths_since_issue_d:<38',
'mths_since_issue_d:38-39',
'mths_since_issue_d:40-41',
'mths_since_issue_d:42-48',
'mths_since_issue_d:49-52',
'mths_since_issue_d:53-64',
'mths_since_issue_d:65-84',
'mths_since_issue_d:>84',
'int_rate:<9.548',
'int_rate:9.548-12.025',
'int_rate:12.025-15.74',
'int_rate:15.74-20.281',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'mths_since_earliest_cr_line:141-164',
'mths_since_earliest_cr_line:165-247',
'mths_since_earliest_cr_line:248-270',
'mths_since_earliest_cr_line:271-352',
'mths_since_earliest_cr_line:>352',
'inq_last_6mths:0',
'inq_last_6mths:1-2',
'inq_last_6mths:3-6',
'inq_last_6mths:>6',
'open_acc:0',
'open_acc:1-3',
'open_acc:4-12',
'open_acc:13-17',
'open_acc:18-22',
'open_acc:23-25',
'open_acc:26-30',
'open_acc:>=31',
'acc_now_delinq:0',
'acc_now_delinq:>=1',
'annual_inc:<20K',
'annual_inc:20K-30K',
'annual_inc:30K-40K',
'annual_inc:40K-50K',
'annual_inc:50K-60K',
'annual_inc:60K-70K',
'annual_inc:70K-80K',
'annual_inc:80K-90K',
'annual_inc:90K-100K',
'annual_inc:100K-125K',
'annual_inc:125K-150K',
'annual_inc:>150K',
'mths_since_last_delinq:Missing',
'mths_since_last_delinq:0-3',
'mths_since_last_delinq:4-30',
'mths_since_last_delinq:31-56',
'mths_since_last_delinq:>=57',
'dti:<=1.4',
'dti:1.4-3.5',
'dti:3.5-7.7',
'dti:7.7-10.5',
'dti:10.5-16.1',
'dti:16.1-20.3',
'dti:20.3-21.7',
'dti:21.7-22.4',
'dti:22.4-35',
'dti:>35',
'mths_since_last_record:Missing',
'mths_since_last_record:0-2',
'mths_since_last_record:3-20',
'mths_since_last_record:21-31',
'mths_since_last_record:32-80',
'mths_since_last_record:81-86'
]


ref_categories = ['grade:G',
'home_ownership:RENT_OTHER_NONE_ANY',
'verification_status:Verified',
'initial_list_status:f',
'addr_state:ND_NE_IA_NV_AL_FL',
'purpose:sm_b__edu__mov__other__house__med__re_e__wedding__vacation',
'term:60',
'emp_length:0',
'mths_since_issue_d:>84',
'int_rate:>20.281',
'mths_since_earliest_cr_line:<140',
'inq_last_6mths:>6',
'open_acc:0',
'acc_now_delinq:0',
'annual_inc:<20K',
'mths_since_last_delinq:0-3',
'dti:>35',
'mths_since_last_record:0-2'
]

In [ ]:
loan_data_preprocessed['PD'] = reg_pd.model.predict_proba(loan_data_preprocessed.loc[:,pd_input_with_cat].drop(ref_categories, axis =1 ))

# EL

In [ ]:
loan_data_preprocessed['EL'] = loan_data_preprocessed['PD']*loan_data_preprocessed['LGD']*loan_data_preprocessed['EAD']


In [ ]:
loan_data_preprocessed[['funded_amnt','PD','LGD','EAD','EL']].head()

,funded_amnt,PD,LGD,EAD,EL
0,4800,0.090072,0.950567,2409.302384,206.282875
1,18000,0.084968,0.865819,14617.549909,1075.367028
2,18000,0.133674,0.919755,10478.181317,1288.265031
3,21600,0.022463,1.000000,14551.083441,326.859805
4,3300,0.135684,0.866632,2496.047982,293.505559


In [ ]:
loan_data_preprocessed['PD'].describe()

count    466285.000000
mean          0.109393
std           0.070851
min           0.007484
25%           0.056071
50%           0.093762
75%           0.147014
max           0.628071
Name: PD, dtype: float64

In [ ]:
loan_data_preprocessed['LGD'].describe()

count    466285.000000
mean          0.931857
std           0.057636
min           0.795735
25%           0.880517
50%           0.912764
75%           1.000000
max           1.000000
Name: LGD, dtype: float64

In [ ]:
loan_data_preprocessed['EAD'].describe()

count    466285.000000
mean      10790.429895
std        6914.123841
min         187.295633
25%        5487.659087
50%        9195.015815
75%       14657.571043
max       35000.000000
Name: EAD, dtype: float64

In [ ]:
loan_data_preprocessed['EL'].describe()

count    466285.000000
mean       1093.531320
std        1111.689300
min          10.231187
25%         355.527660
50%         710.832275
75%        1422.629169
max       11899.656477
Name: EL, dtype: float64

In [ ]:
print("Expected Loss at portfolio level =",loan_data_preprocessed['EL'].sum())

Expected Loss at portfolio level = 509897251.71244735


In [ ]:
print("Total Expected Loss as a proportion of total funded amount for all loans =",loan_data_preprocessed['EL'].sum() / loan_data_preprocessed['funded_amnt'].sum())

Total Expected Loss as a proportion of total funded amount for all loans = 0.07651459161496356
